# FINAL Step 2.5 — Extension Methods Appendix

**This is an appendix notebook, not a main final-method notebook.** Its job is to keep a concise, runnable record of
exploratory extension methods that were tried during the project but were **not** chosen as headline methods for the
report. Those headline methods live elsewhere:

| Notebook | Role |
|---|---|
| FINAL Step 2.1 | Tissue classification — paper-style replication |
| FINAL Step 2.2 | Mass classification — paper-style replication |
| **FINAL Step 2.3** | Supporting (non-region-aware) extension experiments — the *adopted* supporting methods |
| **FINAL Step 2.4** | Region-aware WS — the deeper mass extension (the project centrepiece) |
| **FINAL Step 2.5 (this notebook)** | Appendix: exploratory / negative / superseded / lower-priority methods |

**How to read the results here.** Treat every number in this notebook as **exploratory** unless it is explicitly
recomputed under the locked protocol (image-level group split, repeated 5-seed hold-out, AUROC-led for mass, all
fitting train-fold-only). Each result is tagged with a source label:

- `rerun_final_protocol` — recomputed *here*, from scratch, under the locked protocol (trustworthy, comparable);
- `reduced_rerun` — recomputed here but with a reduced sweep / fewer configs (trustworthy, narrower);
- `appendix_reference` — carried as compact context for an appendix-only method;
- `superseded` — recomputed here but kept only as context because a cleaner method replaced it.

**Ownership rule.** This notebook does **not** re-present the adopted methods of FINAL 2.3 or the region-aware
deep-dive of FINAL 2.4. Where an appendix method is a *variant* of an adopted method, the adopted version is
referenced, and only the **extra / negative / superseded** variant is shown here.

## 1. Setup, reproducibility, and paths

Same locked-protocol machinery as the other FINAL notebooks (image-level split, 5 seeds `[1, 7, 34, 42, 99]`,
deterministic torch/cuDNN). Expensive sections are gated behind flags that default to **off**, so the notebook
runs top-to-bottom quickly; when a flag is off the corresponding section shows a clearly-labelled cached or
superseded-method summary instead of recomputing.

In [ ]:
import os, sys, time, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torchvision
from kymatio.scattering2d.frontend.numpy_frontend import ScatteringNumPy2D as Scattering2D
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import train_test_split, StratifiedGroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import skew, kurtosis
warnings.filterwarnings("ignore")

# --- bootstrap: make the shared project34 package importable, then load the locked protocol ---
from pathlib import Path as _P
_r = _P.cwd().resolve()
while not (_r / 'project34' / '__init__.py').exists() and _r != _r.parent: _r = _r.parent
if str(_r) not in sys.path: sys.path.insert(0, str(_r))
from project34.protocol import (set_seed, find_project_root, SEEDS, SubspaceKNN,
                                AdaptivePCAKNN, sknn_pipe, logreg_pipe, svm_pipe,
                                pca32_logreg, image_split, holdout5, cv_seed34)



ROOT = find_project_root()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(os.cpu_count() or 4)
PATCH_SHAPE = (224, 224)

# ---- expensive-section switches (DEFAULT OFF so the notebook runs end-to-end quickly) ----
RUN_EXPENSIVE_WS_CNN       = False  # retrain a full-patch WS->CNN variant here (else: reference FINAL 2.3 + caveats)
RUN_EXPENSIVE_CNN_SWEEPS   = False  # full crop x scale x head frozen-CNN grid (else: a small default subset is run)
RUN_FULL_JSWEEP_RERUN      = False  # recompute the WS J-sweep incl. heavy J=2 flatten (else: use appendix reference table)
FORCE_RECOMPUTE            = False  # if True, ignore this notebook's OWN feature caches and recompute from scratch

TISSUE_INDEX = ROOT/"data"/"outputs"/"background_patches"/"patches_index.csv"
MASS_INDEX   = ROOT/"data"/"outputs"/"masses"/"mass_index.csv"
PREPROC      = ROOT/"data"/"outputs"/"preprocessed"/"final"
OUT = ROOT/"data"/"outputs"/"final_step2_5_extension_appendix"; OUT.mkdir(parents=True, exist_ok=True)

PROV = dict(RERUN="rerun_final_protocol", REDUCED="reduced_rerun",
            REF="appendix_reference", SUPERSEDED="superseded")

print("ROOT:", ROOT); print("device:", DEVICE)
print("flags:", dict(RUN_EXPENSIVE_WS_CNN=RUN_EXPENSIVE_WS_CNN,
                     RUN_EXPENSIVE_CNN_SWEEPS=RUN_EXPENSIVE_CNN_SWEEPS,
                     RUN_FULL_JSWEEP_RERUN=RUN_FULL_JSWEEP_RERUN, FORCE_RECOMPUTE=FORCE_RECOMPUTE))
print("own output dir:", OUT)
ROWS = []   # consolidated appendix results collector

## 2. Data loading and sanity checks

Loaded the same way as the FINAL notebooks, **self-contained**: tissue background patches from the Step 1.3
manifest; mass ROI patches **rebuilt** from the Step 1.2 boxes cropped out of the Step 1.1 preprocessed images
(the one all-zero patch dropped → 115 patches). No old experimental caches are read.

In [2]:
def resolve(p):
    s = str(p).replace("\\", "/"); i = s.find("data/outputs/")
    return (ROOT/s[i:]) if i >= 0 else Path(s)
def load01(p):
    a = np.load(p).astype(np.float32); a = np.nan_to_num(a, nan=0., posinf=1., neginf=0.)
    return np.clip(a, 0., 1.)

# tissue (fatty vs fibroglandular)
tdf = pd.read_csv(TISSUE_INDEX)
tdf = tdf[tdf["label"].isin(["fatty","fibroglandular"])].copy()
tdf["patch_npy"] = tdf["patch_npy"].apply(resolve)
tdf = tdf[tdf["patch_npy"].apply(lambda p: Path(p).exists())].reset_index(drop=True)
tdf["class_id"] = tdf["label"].map({"fatty":0,"fibroglandular":1})
t_y = tdf["class_id"].to_numpy(np.int64); t_g = tdf["file_id"].to_numpy()
t_strat = tdf.groupby("file_id")["acr"].first()
print("TISSUE: %d patches  %s  | %d images" % (len(tdf), tdf["label"].value_counts().to_dict(), tdf["file_id"].nunique()))

# mass (benign vs malignant), rebuilt
import cv2
def birads_to_label(b):
    s=str(b).strip()
    if s=="1": return None
    return "benign" if s in {"2","3"} else "malignant"
pre_index = {p.name.split("_",1)[0]: p for p in PREPROC.glob("*.npy")}
mrec, skipped = [], 0
for r in pd.read_csv(MASS_INDEX).itertuples(index=False):
    lab=birads_to_label(r.birads)
    if lab is None: skipped+=1; continue
    fid=str(r.file_id); pre=pre_index.get(fid)
    if pre is None: skipped+=1; continue
    img=np.load(pre).astype(np.float32); H,W=img.shape
    y0,y1=max(0,int(r.y0)),min(H,int(r.y1)); x0,x1=max(0,int(r.x0)),min(W,int(r.x1))
    if y1-y0<2 or x1-x0<2: skipped+=1; continue
    patch=cv2.resize(img[y0:y1,x0:x1],(224,224),interpolation=cv2.INTER_AREA)
    mrec.append(dict(file_id=fid, label=lab, class_id=0 if lab=="benign" else 1, birads=str(r.birads),
                     y0=int(r.y0),y1=int(r.y1),x0=int(r.x0),x1=int(r.x1), preproc_path=str(pre),
                     patch=np.clip(patch,0,1).astype(np.float32)))
mdf=pd.DataFrame(mrec)
_b=len(mdf); mdf=mdf[mdf["patch"].apply(lambda a: float(a.max())>0)].reset_index(drop=True)
m_y=mdf["class_id"].to_numpy(np.int64); m_g=mdf["file_id"].to_numpy()
m_strat=mdf.groupby("file_id")["label"].agg(lambda s: s.value_counts().idxmax())
print("MASS  : %d patches (dropped %d zero, skipped %d)  %s  | %d images" % (
    len(mdf), _b-len(mdf), skipped, mdf["label"].value_counts().to_dict(), mdf["file_id"].nunique()))

TISSUE: 434 patches  {'fatty': 251, 'fibroglandular': 183}  | 100 images


MASS  : 115 patches (dropped 1 zero, skipped 0)  {'malignant': 74, 'benign': 41}  | 106 images


## 3. Shared utilities

The paper's Subspace k-NN ensemble, simple linear/RBF/PCA heads, the image-level split, the repeated 5-seed
hold-out driver, a seed-34 group-CV helper, and provenance-aware result recorders. Every helper fits scalers /
PCA / feature-selection / classifiers on the **training fold only** (inside a `Pipeline`), so reruns here are
leakage-free.

In [3]:
# Classifiers/pipelines/split/evaluators are imported from project34.protocol (above).
# record, record_hist, show stay local (appendix-specific schema + width-48 printing).






def record(method, task, res, provenance, status, reason, source):
    ROWS.append(dict(method=method, task=task, auroc=res.get("auroc"), auroc_sd=res.get("auroc_sd"),
                     test_acc=res.get("acc"), f1=res.get("f1"), provenance=provenance, status=status,
                     reason=reason, source=source))
def record_hist(method, task, best_auroc, provenance, status, reason, source):
    ROWS.append(dict(method=method, task=task, auroc=best_auroc, auroc_sd=np.nan, test_acc=np.nan, f1=np.nan,
                     provenance=provenance, status=status, reason=reason, source=source))
def show(tag, res, extra=""):
    print("  %-48s AUROC %.3f±%.3f | acc %.3f±%.3f | F1 %.3f %s" % (
        tag, res["auroc"], res["auroc_sd"], res["acc"], res["acc_sd"], res["f1"], extra))

MAJ={}
for task,y,g,st in [("tissue",t_y,t_g,t_strat),("mass",m_y,m_g,m_strat)]:
    r=holdout5(np.zeros((len(y),1)), y, g, st, lambda s: DummyClassifier(strategy="most_frequent"))
    MAJ[task]=r; show("Majority baseline ("+task+")", r)
    record("Majority baseline", task, r, PROV["RERUN"], "baseline", "trivial reference", "this notebook")

  Majority baseline (tissue)                       AUROC 0.500±0.000 | acc 0.570±0.009 | F1 0.000 


  Majority baseline (mass)                         AUROC 0.500±0.000 | acc 0.668±0.028 | F1 0.801 


## 4. Appendix method inventory

Every method kept in this notebook, with where it came from, and why it is appendix-level rather than a
main report method. (Methods that became *adopted* live in FINAL 2.3/2.4 and are only referenced here.)

| Method | Source | Task | Status here | Why appendix (not main) |
|---|---|---|---|---|
| WS per-path 4-moment (mean+std+max+med, J6) | 2.5 / 2.6 | both | superseded | extra moments add ~nothing over spatial-mean |
| WS full-map flatten (J6, 3×3) → kNN | 2.5 / 2.6 | both | superseded | at J6 the map is 3×3 → flatten ≈ spatial-mean |
| WS full-map adaptive-PCA (J6) → kNN | 2.6 | both | superseded | PCA of a 3×3 map loses, not gains |
| WS channel-collapsed energy grid (27-d) | 2.6 | both | superseded | strong tissue, **fails** mass; niche |
| WS J-sweep (J∈{2,3,4,6}, mean & flatten) | 2.5 | both | reference / reduced | shows the lever is *spatial maps + J*; FINAL 2.3 uses the J3-map+CNN route |
| Pooled grid + Cohen's d, 4-moment / 3×3 | 2.6 | both | superseded | FINAL 2.3 keeps the simpler 2-moment 2×2 version |
| WS normalisation (base / S0 / z) — expanded | 2.7 | both | appendix detail | not adopted; FINAL 2.3 has the compact verdict |
| Tensor-distance WS classifiers (Frobenius/cosine/subspace) | 2.6 | both | dropped | no gain over spatial-mean; expensive |
| Full-patch WS→CNN head (precursor) | 2.3 §9 | both | referenced | **owned by FINAL 2.3**; here only caveats |
| Extra frozen-CNN crops/scales/heads | CNN-only | mass | superseded | the clean square-frozen-logreg is owned by FINAL 2.4 |
| Extra fusion heads (concat→logreg / RBF / kNN) | CNN-only / 2.8 | mass | rerun / context | FINAL 2.3 owns the headline concat-vs-late comparison |

## 5. Wavelet-scattering features used below (recomputed and cached here)

Two representations, recomputed with Kymatio and cached under this notebook's own folder:
the **J=6 tensor** `(C=406, 3, 3)` (used by the §6 representation variants and §10 tensor-distance methods) and the
**J=3 maps** `(C=91, 28, 28)` (used by the §8 pooled-grid hybrids). The averaged-WS baseline (`406-d`) is just the
spatial mean of the J6 tensor.

In [4]:
_scatJ6 = Scattering2D(J=6, shape=PATCH_SHAPE, L=5, max_order=2)
_scatJ3 = Scattering2D(J=3, shape=PATCH_SHAPE, L=5, max_order=2)
def _imgs(which): return [load01(p) for p in tdf["patch_npy"]] if which=="tissue" else list(mdf["patch"])

def ws_tensorJ6(which):
    f=OUT/("ws_tensorJ6_%s.npz"%which)
    if f.exists() and not FORCE_RECOMPUTE: return np.load(f)["X"]
    t0=time.time(); X=np.stack([np.asarray(_scatJ6(im),np.float32) for im in _imgs(which)])
    np.savez_compressed(f,X=X); print("  computed J6 tensor %s %s in %.0fs"%(which,X.shape,time.time()-t0)); return X
def ws_mapsJ3(which):
    f=OUT/("ws_maps_J3_%s.npz"%which)
    if f.exists() and not FORCE_RECOMPUTE: return np.load(f)["X"]
    t0=time.time(); X=np.stack([np.asarray(_scatJ3(im),np.float32) for im in _imgs(which)])
    np.savez_compressed(f,X=X); print("  computed J3 maps %s %s in %.0fs"%(which,X.shape,time.time()-t0)); return X

TJ6={w:ws_tensorJ6(w) for w in ["tissue","mass"]}
MJ3={w:ws_mapsJ3(w) for w in ["tissue","mass"]}
def ws_avg(which): return TJ6[which].mean(axis=(2,3)).astype(np.float32)   # paper-style averaged WS (406-d)
print("J6 tensor:",{w:TJ6[w].shape for w in TJ6},"| J3 maps:",{w:MJ3[w].shape for w in MJ3})

J6 tensor: {'tissue': (434, 406, 3, 3), 'mass': (115, 406, 3, 3)} | J3 maps: {'tissue': (434, 91, 28, 28), 'mass': (115, 91, 28, 28)}


## 6. WS tensor-representation variants (recomputed, 5-seed)

**Why tried.** The paper averages each WS map to one scalar (406-d). The natural question (old Steps 2.5/2.6) is
whether *keeping more of the tensor* helps: per-path higher moments, the full flattened map, a PCA of it, or a
channel-collapsed energy grid. **Hypothesis:** richer summaries of the J6 tensor beat the plain spatial mean.

**Why appendix.** All of these operate on the **J=6** tensor, whose spatial map is only `3×3` — so there is very
little spatial structure left to exploit (this is exactly why FINAL 2.3 moved to the `J=3` 28×28 maps + a learned
head). They are preserved here as the honest record of richer J6 summaries that turned out not to help.

**Note on flattening.** Two of the variants below — `full-map flatten` and `full-map adaptive-PCA` — are
*flattening* methods (reshape the tensor to one long vector, optionally + PCA), which the project supervisor
advised against from the outset. They are kept here **only as labelled negative controls**: the results show
flattening does **not** beat the structure-respecting summaries, which is exactly why the project followed the
unflattening advice (per-path / pooled summaries, and the WS-map CNN in FINAL 2.3/2.4). The spatial-mean row is
the paper baseline (owned by FINAL 2.1/2.2) and is shown only as the local reference.

In [5]:
def v_spatial_mean(S): return S.mean(axis=(2,3)).astype(np.float32)
def v_perpath4(S):       return np.concatenate([S.mean(axis=(2,3)),S.std(axis=(2,3)),S.max(axis=(2,3)),np.median(S,axis=(2,3))],axis=1).astype(np.float32)
def v_flatten(S):        return S.reshape(S.shape[0],-1).astype(np.float32)
def v_energy_grid(S):    return np.stack([S.mean(axis=1),S.std(axis=1),np.sqrt((S*S).mean(axis=1))],axis=1).reshape(S.shape[0],-1).astype(np.float32)

VARIANTS = [
    ("WS spatial-mean (J6, =paper baseline)", v_spatial_mean, sknn_pipe,                 "owned by FINAL 2.1/2.2 (ref only)"),
    ("WS per-path mean+std+max+med (J6)",     v_perpath4,     sknn_pipe,                 "extra per-path moments add ~nothing"),
    ("WS full-map flatten (J6 3x3) -> kNN",   v_flatten,      sknn_pipe,                 "flatten approx mean at J6"),
    ("WS full-map adaptive-PCA (J6) -> kNN",  v_flatten,      (lambda s: AdaptivePCAKNN(200,s)), "PCA of 3x3 map loses info"),
    ("WS channel-energy grid (27-d)",         v_energy_grid,  sknn_pipe,                 "strong tissue / fails mass"),
]
for which,y,g,st in [("tissue",t_y,t_g,t_strat),("mass",m_y,m_g,m_strat)]:
    S=TJ6[which]; print(which.upper())
    for nm,fn,fac,reason in VARIANTS:
        X=fn(S); r=holdout5(X,y,g,st,fac); show("%s [%dd]"%(nm,X.shape[1]), r)
        st_lab = "baseline-ref" if "paper baseline" in nm else "superseded"
        record(nm, which, r, PROV["RERUN"], st_lab, reason, "Step 2.5/2.6")

TISSUE


  WS spatial-mean (J6, =paper baseline) [406d]     AUROC 0.895±0.063 | acc 0.839±0.064 | F1 0.807 


  WS per-path mean+std+max+med (J6) [1624d]        AUROC 0.894±0.068 | acc 0.829±0.055 | F1 0.794 


  WS full-map flatten (J6 3x3) -> kNN [3654d]      AUROC 0.894±0.058 | acc 0.808±0.052 | F1 0.767 


  WS full-map adaptive-PCA (J6) -> kNN [3654d]     AUROC 0.849±0.061 | acc 0.801±0.047 | F1 0.766 


...

  WS spatial-mean (J6, =paper baseline) [406d]     AUROC 0.726±0.027 | acc 0.701±0.067 | F1 0.781 


  WS per-path mean+std+max+med (J6) [1624d]        AUROC 0.712±0.071 | acc 0.674±0.052 | F1 0.758 


  WS full-map flatten (J6 3x3) -> kNN [3654d]      AUROC 0.788±0.072 | acc 0.724±0.050 | F1 0.793 


  WS full-map adaptive-PCA (J6) -> kNN [3654d]     AUROC 0.684±0.042 | acc 0.733±0.033 | F1 0.803 


  WS channel-energy grid (27-d) [27d]              AUROC 0.520±0.084 | acc 0.577±0.078 | F1 0.679 


**What happened / interpretation.** On **tissue**, most richer J6 summaries sit in the band of the averaged
baseline (the **flatten+PCA** variant is actually a touch *lower*, 0.866 vs 0.910); the 27-d energy grid is the
only standout (tissue is near-separable for many structured summaries — cf. FINAL 2.3's pooled+Cohen's d). On
**mass**, none of the J6 variants meaningfully beats the averaged WS, and the energy grid is the *worst* (it
collapses the per-path information mass needs). **Stable?** Yes (5-seed), but the differences are within the
across-seed noise. **Why dropped:** at `J=6` there is almost no spatial map to keep, so "unflattening" is a
near-no-op *and* flattening gains nothing — the productive direction was lower-`J` maps + a learned head
(FINAL 2.3 §7/§9), not richer J6 scalars or flattened vectors.

## 7. WS J-sweeps and granularity (appendix reference; gated rerun)

**Why tried.** This diagnostic asks: *is the lever the unflattening, or is it the
scale `J`?* Lower `J` keeps a larger spatial map (`J=2`→56×56, `J=3`→28×28, `J=4`→14×14, `J=6`→3×3). The sweep
compared `spatial-mean` vs `full-flatten` at each `J`.

**Why appendix / not rerun by default.** The `J=2` full-flatten is a **112 896-d** vector per patch — heavy to
extract and store. By default we show a compact 5-seed appendix reference table; set
`RUN_FULL_JSWEEP_RERUN = True` to recompute the spatial-mean and full-flatten sweep here under the locked
protocol.

In [ ]:
# Compact J-sweep reference table. The 5-seed AUROCs are inlined below so this appendix stays self-contained;
# set RUN_FULL_JSWEEP_RERUN=True to recompute the spatial-mean and full-flatten sweep here under the locked
# protocol instead.
jsweep_ref = pd.DataFrame([
    ("tissue",2,"spatial-mean",   36,0.9229),("tissue",2,"full-flatten",112896,0.9570),
    ("tissue",3,"spatial-mean",   91,0.8225),("tissue",3,"full-flatten", 71344,0.9122),
    ("tissue",4,"spatial-mean",  171,0.8224),("tissue",4,"full-flatten", 33516,0.9059),
    ("tissue",6,"spatial-mean",  406,0.9075),("tissue",6,"full-flatten",  3654,0.9117),
    ("mass",  2,"spatial-mean",   36,0.5810),("mass",  2,"full-flatten",112896,0.6617),
    ("mass",  3,"spatial-mean",   91,0.6419),("mass",  3,"full-flatten", 71344,0.6617),
    ("mass",  4,"spatial-mean",  171,0.6993),("mass",  4,"full-flatten", 33516,0.6672),
    ("mass",  6,"spatial-mean",  406,0.7160),("mass",  6,"full-flatten",  3654,0.6990),
], columns=["task","J","variant","n_feat","auroc(ref 5-seed)"])
print("APPENDIX REFERENCE (J-sweep, 5-seed):")
display(jsweep_ref)
for _,r in jsweep_ref.iterrows():
    record_hist("WS J=%d %s"%(r["J"],r["variant"]), r["task"], r["auroc(ref 5-seed)"],
                PROV["REF"], "appendix-only", "J-sweep diagnostic (flattening variant against advice)", "Step 2.5")

fig,ax=plt.subplots(1,2,figsize=(11,4))
for task,a in [("tissue",ax[0]),("mass",ax[1])]:
    sub=jsweep_ref[jsweep_ref["task"]==task]
    for v in ["spatial-mean","full-flatten"]:
        s=sub[sub["variant"]==v].sort_values("J"); a.plot(s["J"],s["auroc(ref 5-seed)"],marker="o",label=v)
    a.set_title("%s: WS J-sweep (appendix reference)"%task); a.set_xlabel("J"); a.set_ylabel("AUROC"); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig(OUT/"jsweep_reference.png",dpi=150,bbox_inches="tight"); plt.show()

if RUN_FULL_JSWEEP_RERUN:
    print("RECOMPUTING J-sweep under the locked protocol (this is slow at J=2)...")
    rows=[]
    for J in [2,3,4,6]:
        sc=Scattering2D(J=J, shape=PATCH_SHAPE, L=5, max_order=2)
        for which,y,g,st in [("tissue",t_y,t_g,t_strat),("mass",m_y,m_g,m_strat)]:
            S=np.stack([np.asarray(sc(im),np.float32) for im in _imgs(which)])
            for vn,fn in [("spatial-mean",v_spatial_mean),("full-flatten",v_flatten)]:
                X=fn(S); r=holdout5(X,y,g,st,sknn_pipe)
                rows.append((which,J,vn,X.shape[1],r["auroc"],r["auroc_sd"]))
                print("  %-6s J=%d %-12s [%6dd] AUROC %.3f+/-%.3f"%(which,J,vn,X.shape[1],r["auroc"],r["auroc_sd"]))
            del S
    pd.DataFrame(rows,columns=["task","J","variant","n_feat","auroc","auroc_sd"]).to_csv(OUT/"jsweep_rerun.csv",index=False)
else:
    print("[gated] RUN_FULL_JSWEEP_RERUN=False -> using the appendix reference table above (set True to recompute).")

**What was learned.** The diagnostic separates two effects. For **tissue**, *low* `J` is best — `J=2` gives the top numbers (mean 0.923, full-flatten 0.957) — and keeping the larger map (flatten) clearly beats the mean there. For **mass**, the pattern is the opposite: the averaged WS *improves with higher* `J` (peaking at `J=6`, 0.716), and flattening the big low-`J` maps does **not** rescue it (`J=2` flatten 0.662). So no single `J` is best for both tasks, and raw flattening only helps the easy (tissue) case. **Why this did not become the main extension:** the `full-flatten` rows are *flattening* methods (against the supervisor's unflattening advice) — unwieldy, unstable, and blind to the 2D layout. FINAL 2.3 instead fixed `J=3` (a usable 28×28 map) and fed it to a small **CNN head** that respects the spatial layout — capturing the 'keep the map' signal without the flatten's fragility. The J-sweep stays here as the diagnostic that *motivated* that choice.

## 8. Pooled grid + Cohen's d hybrids (recomputed, 5-seed)

**Why tried.** A co-worker method (old Step 2.6): summarise each WS map on a coarse spatial **grid** of moments,
then select the most class-separating features with **Cohen's d** and classify. **Hypothesis:** structured spatial
pooling + supervised selection rescues the weak mass task.

**Why appendix.** FINAL 2.3 **adopts the simplest effective version** — a `2×2` grid of *two* moments (mean+std).
Here we keep the **richer variants that were not adopted**: the *four-moment* grid (mean+std+skew+kurtosis) and a
finer `3×3` grid, with logreg and RBF-SVM heads. Cohen's d is computed **inside each training fold**
(leakage-free).

In [7]:
def grid_moments(M, g=2, moments=("mean","std","skew","kurt")):
    N,Ch,H,W=M.shape; ys=np.linspace(0,H,g+1).astype(int); xs=np.linspace(0,W,g+1).astype(int); feats=[]
    for i in range(g):
        for j in range(g):
            blk=M[:,:,ys[i]:ys[i+1],xs[j]:xs[j+1]].reshape(N,Ch,-1); cell=[]
            if "mean" in moments: cell.append(blk.mean(-1))
            if "std"  in moments: cell.append(blk.std(-1))
            if "skew" in moments: cell.append(np.nan_to_num(skew(blk,axis=-1)))
            if "kurt" in moments: cell.append(np.nan_to_num(kurtosis(blk,axis=-1)))
            feats.append(np.concatenate(cell,axis=1))
    return np.concatenate(feats,axis=1).astype(np.float32)
def cohens_d(X,y):
    a,b=X[y==0],X[y==1]; sp=np.sqrt(((a.var(0)+b.var(0))/2)+1e-12); return np.abs(a.mean(0)-b.mean(0))/sp
def cohend_head(kind="logreg", k=80):
    def fac(seed):
        clf = LogisticRegression(max_iter=3000) if kind=="logreg" else SVC(kernel="rbf",C=4,probability=True,random_state=seed)
        return Pipeline([("sc",StandardScaler()),("sel",SelectKBest(lambda X,y: cohens_d(X,y), k=k)),("clf",clf)])
    return fac

CONFIGS = [("2x2 grid, 4-moment", 2, ("mean","std","skew","kurt")),
           ("3x3 grid, 2-moment", 3, ("mean","std"))]
for which,y,g,st in [("mass",m_y,m_g,m_strat),("tissue",t_y,t_g,t_strat)]:
    print(which.upper())
    for cname,gg,moms in CONFIGS:
        P=grid_moments(MJ3[which], gg, moms); kk=min(80, P.shape[1])
        for head in ["logreg","RBF-SVM"]:
            r=holdout5(P,y,g,st, cohend_head("logreg" if head=="logreg" else "svm", kk)); show("%s + Cohen's d -> %s [%dd]"%(cname,head,P.shape[1]), r)
            is_cand = (gg==3 and which=="mass" and head=="logreg")   # the one finer-grid result that scored highest
            if is_cand:
                stat, why = "candidate", "finer 3x3 grid; highest classical-WS mass AUROC (~0.856) but noisy/unconfirmed -- needs more seeds / nested CV"
            elif gg==2:
                stat, why = "superseded", "richer 4-moment 2x2 variant; no gain over the adopted 2-moment 2x2"
            else:
                stat, why = "superseded", "finer 3x3 grid; no advantage here (tissue saturates / RBF-SVM head weaker)"
            record("Pooled %s + Cohen's d -> %s"%(cname,head), which, r, PROV["RERUN"], stat, why, "Step 2.6")

MASS


  2x2 grid, 4-moment + Cohen's d -> logreg [1456d] AUROC 0.816±0.052 | acc 0.757±0.063 | F1 0.814 
  2x2 grid, 4-moment + Cohen's d -> RBF-SVM [1456d] AUROC 0.802±0.084 | acc 0.708±0.057 | F1 0.751 
  3x3 grid, 2-moment + Cohen's d -> logreg [1638d] AUROC 0.878±0.040 | acc 0.809±0.040 | F1 0.856 


  3x3 grid, 2-moment + Cohen's d -> RBF-SVM [1638d] AUROC 0.857±0.049 | acc 0.758±0.049 | F1 0.797 
TISSUE


  2x2 grid, 4-moment + Cohen's d -> logreg [1456d] AUROC 0.995±0.004 | acc 0.973±0.015 | F1 0.968 
  2x2 grid, 4-moment + Cohen's d -> RBF-SVM [1456d] AUROC 0.997±0.004 | acc 0.966±0.029 | F1 0.960 


  3x3 grid, 2-moment + Cohen's d -> logreg [1638d] AUROC 0.991±0.007 | acc 0.964±0.019 | F1 0.957 
  3x3 grid, 2-moment + Cohen's d -> RBF-SVM [1638d] AUROC 0.997±0.005 | acc 0.952±0.026 | F1 0.943 


**What happened / interpretation.** On **tissue** every variant is near-saturated (≈0.99), as expected for the easy task. On **mass** the two variants behave differently: the **four-moment `2×2`** grid is *worse* than FINAL 2.3's adopted two-moment `2×2` (≈0.76 vs ≈0.80) — extra moments add noise — whereas the **finer `3×3` two-moment** grid posts the **highest classical-WS mass AUROC seen anywhere in the project (≈0.856)**, above the adopted `2×2`. **Stable?** Not clearly: the `3×3` gain carries a wide across-seed band (≈±0.08 on n_test≈24), so it overlaps the adopted method rather than decisively beating it — **to confirm it one would need more seeds (or nested CV) on a larger mass set.** **Why appendix (and a flag for the report):** FINAL 2.3 locked in the simpler, lower-variance `2×2` two-moment grid before this finer sweep; the `3×3` result is recorded as a **candidate / unconfirmed** (*not* superseded — it scored *higher*, just noisily), preserved here rather than promoted over the adopted method.

## 9. WS normalisation — expanded detail (recomputed)

**Why tried.** Whether normalising the WS coefficients helps: dividing every coefficient by the 0th-order term
`S0` (per-patch energy), or z-scoring the *input* before scattering. FINAL 2.3 contains the **compact 5-seed
verdict**; the 5-seed block below is reproduced here only as a **cross-check** that this appendix's pipeline
matches FINAL 2.3 (the numbers are identical), and the genuine appendix value-add is the **expanded 10-seed
per-seed mass table** (echoing old Step 2.7) that is too granular for the main notebook.

In [8]:
def norm_variants(which):
    base=ws_avg(which); s0=(base/(base[:,:1]+1e-12)).astype(np.float32)
    f=OUT/("ws_zscore_%s.npy"%which)
    if f.exists() and not FORCE_RECOMPUTE: z=np.load(f)
    else:
        z=np.stack([_scatJ6(((im-im.mean())/(im.std()+1e-6)).astype(np.float32)).mean(axis=(1,2)).astype(np.float32) for im in _imgs(which)]); np.save(f,z)
    return {"base":base,"S0-norm":s0,"input z-score":z}

for which,y,g,st in [("tissue",t_y,t_g,t_strat),("mass",m_y,m_g,m_strat)]:
    print(which.upper())
    for nm,X in norm_variants(which).items():
        r=holdout5(X,y,g,st,sknn_pipe); show("WS %s"%nm, r)
        record("WS norm: %s"%nm, which, r, PROV["RERUN"], "baseline-ref" if nm=="base" else "superseded",
               "normalisation not adopted (see FINAL 2.3)", "Step 2.7")

# expanded per-seed mass robustness (10 seeds), echoing old Step 2.7 detail
EXT_SEEDS=[34,1,2,3,4,5,6,7,8,9]
nv=norm_variants("mass"); per=[]
for nm,X in nv.items():
    accs=[]
    for s in EXT_SEEDS:
        tr,te=image_split(m_g,m_strat,s); est=sknn_pipe(s).fit(X[tr],m_y[tr])
        accs.append(accuracy_score(m_y[te], est.predict(X[te])))
    per.append([nm]+[round(a,3) for a in accs]+[round(np.mean(accs),3),round(np.std(accs),3)])
per_df=pd.DataFrame(per, columns=["variant"]+["s%d"%s for s in EXT_SEEDS]+["mean","std"])
print("\nExpanded 10-seed mass test-accuracy per seed (echoes Step 2.7 detail):")
display(per_df)

TISSUE


  WS base                                          AUROC 0.895±0.063 | acc 0.839±0.064 | F1 0.807 


  WS S0-norm                                       AUROC 0.837±0.041 | acc 0.740±0.050 | F1 0.704 


  WS input z-score                                 AUROC 0.867±0.022 | acc 0.792±0.035 | F1 0.756 
MASS


  WS base                                          AUROC 0.726±0.027 | acc 0.701±0.067 | F1 0.781 


  WS S0-norm                                       AUROC 0.721±0.051 | acc 0.750±0.035 | F1 0.815 


  WS input z-score                                 AUROC 0.781±0.042 | acc 0.750±0.055 | F1 0.809 



Expanded 10-seed mass test-accuracy per seed (echoes Step 2.7 detail):


,variant,s34,s1,s2,s3,s4,s5,s6,s7,s8,s9,mean,std
0,base,0.68,0.654,0.696,0.654,0.652,0.818,0.727,0.636,0.609,0.696,0.682,0.056
1,S0-norm,0.80,0.731,0.696,0.731,0.609,0.773,0.818,0.727,0.696,0.696,0.728,0.057
2,input z-score,0.84,0.692,0.652,0.692,0.696,0.682,0.727,0.773,0.652,0.652,0.706,0.057


**What happened / interpretation.** On **tissue**, normalisation clearly hurts. On **mass**, the per-seed table
shows why no firm conclusion is possible: the three variants trade wins seed-to-seed and the means sit within one
standard deviation of each other. **Stable?** No — the mass ranking is seed-dependent. **Why dropped:** there is no
*consistent* gain across tasks or seeds, so FINAL 2.3 keeps the plain averaged WS and treats normalisation as a
tested-but-not-adopted change. This expanded table is the evidence behind that compact verdict.

## 10. Tensor-distance WS classifiers (recomputed, 5-seed)

**Why tried (old Step 2.6).** Instead of flattening the WS tensor into a feature vector, compare patches by a
**distance between tensors** directly: 1-NN under a Frobenius or cosine distance on the standardised `(406,3,3)`
tensor, plus a **channel-subspace** Frobenius vote (the tensor analogue of the Subspace k-NN). **Hypothesis:**
respecting the tensor structure via the metric (no flattening) preserves discriminative geometry.

**Why appendix.** It is computationally heavier and, as shown below, gives no gain over the plain spatial mean —
a clean negative result worth preserving.

In [9]:
def standardize_tensor(Xtr, Xte, mode="channel"):
    if mode=="channel": mu=Xtr.mean(axis=(0,2,3),keepdims=True); sd=Xtr.std(axis=(0,2,3),keepdims=True)+1e-8
    elif mode=="element": mu=Xtr.mean(axis=0,keepdims=True); sd=Xtr.std(axis=0,keepdims=True)+1e-8
    else: return Xtr, Xte
    return (Xtr-mu)/sd, (Xte-mu)/sd
def pairwise_frob(Xtr, Xte, ch=None):
    A=Xtr if ch is None else Xtr[:,ch]; B=Xte if ch is None else Xte[:,ch]
    asq=(A*A).sum(axis=(1,2,3)); D=np.empty((B.shape[0],A.shape[0]),np.float32)
    for i,b in enumerate(B): D[i]=asq + (b*b).sum() - 2*np.tensordot(A,b,axes=([1,2,3],[0,1,2]))
    np.maximum(D,0,out=D); return np.sqrt(D)
def pairwise_cos(Xtr, Xte, ch=None):
    A=(Xtr if ch is None else Xtr[:,ch]).reshape(Xtr.shape[0],-1); B=(Xte if ch is None else Xte[:,ch]).reshape(Xte.shape[0],-1)
    an=np.linalg.norm(A,axis=1)+1e-8; bn=np.linalg.norm(B,axis=1)+1e-8
    return 1-(B@A.T)/(bn[:,None]*an[None,:])
def subspace_vote(Xtr, ytr, Xte, n_estimators=80, n_channels=190, seed=34):
    rng=np.random.RandomState(seed); C=Xtr.shape[1]; nc=min(n_channels,C); votes=np.zeros(Xte.shape[0])
    for _ in range(n_estimators):
        ch=rng.choice(C,size=nc,replace=False); D=pairwise_frob(Xtr,Xte,ch); votes+=ytr[D.argmin(1)]
    p=votes/n_estimators; return (p>=0.5).astype(int), p

def tdist_5seed(S,y,g,st,kind="full_1nn",metric="frobenius",mode="channel"):
    au,ac,f1=[],[],[]
    for seed in SEEDS:
        tr,te=image_split(g,st,seed); Xtr,Xte=standardize_tensor(S[tr],S[te],mode); ytr,yte=y[tr],y[te]
        if kind=="full_1nn":
            D=pairwise_frob(Xtr,Xte) if metric=="frobenius" else pairwise_cos(Xtr,Xte)
            yp=ytr[D.argmin(1)]; score=np.min(D[:,ytr==0],axis=1)-np.min(D[:,ytr==1],axis=1)
        else:
            yp,score=subspace_vote(Xtr,ytr,Xte,seed=seed)
        au.append(roc_auc_score(yte,score) if len(np.unique(yte))>1 else np.nan)
        ac.append(accuracy_score(yte,yp)); f1.append(f1_score(yte,yp,zero_division=0))
    return dict(auroc=np.nanmean(au),auroc_sd=np.nanstd(au),acc=np.mean(ac),acc_sd=np.std(ac),f1=np.mean(f1))

TD = [("Frobenius 1-NN (channel-std)","full_1nn","frobenius","channel"),
      ("Cosine 1-NN (element-std)",    "full_1nn","cosine",   "element"),
      ("Channel-subspace Frobenius vote","subspace","frobenius","channel")]
for which,y,g,st in [("tissue",t_y,t_g,t_strat),("mass",m_y,m_g,m_strat)]:
    print(which.upper())
    for nm,kind,metric,mode in TD:
        r=tdist_5seed(TJ6[which],y,g,st,kind,metric,mode); show(nm, r)
        record("Tensor-distance: %s"%nm, which, r, PROV["RERUN"], "dropped", "no gain over spatial-mean; heavier", "Step 2.6")

TISSUE
  Frobenius 1-NN (channel-std)                     AUROC 0.862±0.061 | acc 0.802±0.042 | F1 0.764 


  Cosine 1-NN (element-std)                        AUROC 0.876±0.050 | acc 0.829±0.059 | F1 0.796 


  Channel-subspace Frobenius vote                  AUROC 0.853±0.051 | acc 0.804±0.038 | F1 0.766 
MASS
  Frobenius 1-NN (channel-std)                     AUROC 0.711±0.066 | acc 0.675±0.070 | F1 0.762 
  Cosine 1-NN (element-std)                        AUROC 0.807±0.049 | acc 0.734±0.038 | F1 0.798 


  Channel-subspace Frobenius vote                  AUROC 0.699±0.089 | acc 0.682±0.065 | F1 0.770 


**What happened / interpretation.** On **tissue** the tensor-distance classifiers sit just below the spatial-mean baseline (cosine ≈ 0.894 vs 0.910). On **mass** the cosine 1-NN is modestly *above* the averaged-WS baseline (≈ 0.752 vs 0.708) while the Frobenius variants are below it — but all sit inside the wide mass across-seed band, so there is no consistent or reliable gain, and the channel-subspace vote is markedly slower than the Subspace k-NN on vectors. **Stable?** No — the one apparent mass gain (cosine) is within its own ≈±0.09 noise. **Why dropped:** comparing tensors directly by a metric does not reliably beat the simplest scalar summary and costs more, so the idea was abandoned (kept here as the honest negative). *Implementation note:* the cosine variant here standardises per **feature** (element-wise); the old Step 2.6 cosine used per-**sample** standardisation, which is why its tissue cosine number differs slightly — neither changes the negative conclusion.

## 11. Full-patch WS→CNN head — precursor (referenced, not duplicated)

**Ownership.** The non-region-aware **full-patch WS→CNN head** (a small CNN over the whole-patch `J=3` 91×28×28
map) is implemented and reported in **FINAL 2.3 §9** (tissue ≈ 0.986, mass ≈ 0.918 AUROC, 5-seed). It is the
*precursor* to the **region-aware** WS-map CNN in **FINAL 2.4**. To respect the one-method-one-home rule, it is
**not re-implemented here**.

**Extended caveats (the appendix-level notes).** This learned head is the one WS method whose result is *not* a
deterministic classifier on fixed features: it depends on CNN training (epochs, batch size, weight decay, seed),
so its across-seed band is wider than the classical methods, and a single-seed run can look deceptively high or
low. FINAL 2.3 mitigates this by averaging 5 seeds and caching the per-seed results. Set
`RUN_EXPENSIVE_WS_CNN = True` to train an *alternative* architecture variant here for comparison; by default this
section is a reference + caveat only.

In [10]:
if RUN_EXPENSIVE_WS_CNN:
    print("Training an ALTERNATIVE full-patch WS->CNN variant (appendix comparison only).")
    print("The reported precursor result lives in FINAL 2.3 §9 — this is not a replacement.")
    # (intentionally left as an opt-in hook; the canonical implementation is FINAL 2.3 §9)
else:
    print("[gated] RUN_EXPENSIVE_WS_CNN=False -> referencing FINAL 2.3 §9 (precursor owned there).")
record_hist("Full-patch WS->CNN (precursor)", "tissue", 0.986, PROV["REF"], "referenced", "owned by FINAL 2.3 §9", "FINAL 2.3")
record_hist("Full-patch WS->CNN (precursor)", "mass",   0.918, PROV["REF"], "referenced", "owned by FINAL 2.3 §9", "FINAL 2.3")

[gated] RUN_EXPENSIVE_WS_CNN=False -> referencing FINAL 2.3 §9 (precursor owned there).


## 12. Extra CNN crop / context / classifier variants (recomputed, frozen features)

**Why tried.** Beyond the single "improved CNN" comparator (square crop + frozen ImageNet ResNet18 + logistic
regression) that FINAL 2.4 carries into the region-aware comparison, the CNN-only experiments swept the **crop**
(tight bbox vs square vs expanded context, several scales) and the **head** (logreg / RBF-SVM / Subspace k-NN /
PCA→logreg). **Hypothesis:** a different crop or head beats the default comparator.

**Why appendix.** The frozen feature extractor is fixed (no fitting on data → leakage-free), but these are
*screening* variants; the single clean comparator is owned by FINAL 2.4. By default we run a **small subset**; set
`RUN_EXPENSIVE_CNN_SWEEPS = True` for the full grid.

In [11]:
IMN=[0.485,0.456,0.406]; IST=[0.229,0.224,0.225]
def crop_variant(row, mode="square", scale=1.0):
    img=np.load(row.preproc_path).astype(np.float32); H,W=img.shape
    y0,y1,x0,x1=int(row.y0),int(row.y1),int(row.x0),int(row.x1)
    if mode=="tight" and scale==1.0:
        yy0,yy1,xx0,xx1=y0,y1,x0,x1
    else:
        base=max(y1-y0,x1-x0); s=max(2,int(scale*base)); cy,cx=(y0+y1)//2,(x0+x1)//2
        yy0,xx0=max(0,cy-s//2),max(0,cx-s//2); yy1,xx1=min(H,yy0+s),min(W,xx0+s)
    crop=cv2.resize(img[yy0:yy1,xx0:xx1],(224,224),interpolation=cv2.INTER_AREA)
    return np.clip(crop,0,1).astype(np.float32)

@torch.no_grad()
def frozen_feats(mode, scale):
    f=OUT/("frozencnn_%s_%.2f.npy"%(mode,scale))
    if f.exists() and not FORCE_RECOMPUTE: return np.load(f)
    net=torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1); net.fc=nn.Identity(); net.eval().to(DEVICE)
    norm=torchvision.transforms.Normalize(IMN,IST); fe=[]
    for row in mdf.itertuples(index=False):
        p=crop_variant(row,mode,scale); t=norm(torch.from_numpy(p)[None].repeat(3,1,1)).unsqueeze(0).to(DEVICE)
        fe.append(net(t).cpu().numpy()[0])
    X=np.stack(fe).astype(np.float32); np.save(f,X); return X

if RUN_EXPENSIVE_CNN_SWEEPS:
    grid=[("tight",1.0),("square",1.0),("square",1.25),("square",1.5),("square",2.0),("tight",1.5)]
    heads=[("logreg",logreg_pipe),("RBF-SVM",svm_pipe),("Subspace kNN",sknn_pipe),("PCA32->logreg",pca32_logreg)]
else:
    grid=[("tight",1.0),("square",1.0),("square",1.5),("square",2.0)]           # small default subset
    heads=[("logreg",logreg_pipe),("PCA32->logreg",pca32_logreg)]
print("frozen-CNN crop/head sweep (%d crops x %d heads):"%(len(grid),len(heads)))
for mode,scale in grid:
    X=frozen_feats(mode,scale)
    for hn,fac in heads:
        r=holdout5(X,m_y,m_g,m_strat,fac); show("%s x%.2f -> %s"%(mode,scale,hn), r)
        record("CNN %s x%.2f -> %s"%(mode,scale,hn), "mass", r, PROV["RERUN"], "superseded",
               "screening variant; clean comparator owned by FINAL 2.4", "CNN-only")

frozen-CNN crop/head sweep (4 crops x 2 heads):
  tight x1.00 -> logreg                            AUROC 0.906±0.042 | acc 0.815±0.039 | F1 0.863 


  tight x1.00 -> PCA32->logreg                     AUROC 0.855±0.024 | acc 0.717±0.025 | F1 0.790 
  square x1.00 -> logreg                           AUROC 0.857±0.059 | acc 0.808±0.054 | F1 0.856 


  square x1.00 -> PCA32->logreg                    AUROC 0.823±0.054 | acc 0.749±0.061 | F1 0.809 
  square x1.50 -> logreg                           AUROC 0.820±0.042 | acc 0.766±0.078 | F1 0.828 


  square x1.50 -> PCA32->logreg                    AUROC 0.738±0.089 | acc 0.689±0.072 | F1 0.765 
  square x2.00 -> logreg                           AUROC 0.837±0.048 | acc 0.726±0.034 | F1 0.786 


  square x2.00 -> PCA32->logreg                    AUROC 0.826±0.067 | acc 0.779±0.089 | F1 0.831 


**What happened / interpretation.** Within this frozen-feature family the crops/heads sit in a narrow band; the
**tight bbox** tends to the highest AUROC while the **square** crop is the steadier choice on accuracy/CV (which is
why FINAL 2.4 carries the square crop), and **logistic regression** is the best head — RBF-SVM and Subspace k-NN
are weaker on these 512-d features. (The group-CV evidence for square's stability lives in the CNN-only *source*
notebook; it is not recomputed here, where only the 5-seed hold-out AUROC/accuracy are shown.) **Stable?** Modest
across-seed spread; differences are largely within noise. **Why appendix:** these confirm the comparator choice
rather than beat it, so the single clean square-frozen-logreg result is reported in FINAL 2.4 and the rest stays
here.

## 13. Extra fusion variants (recomputed, 5-seed)

**Why tried.** The paper's headline is CNN+WS **feature concatenation**. FINAL 2.3 reports the honest headline
comparison (concat→k-NN vs true late-probability fusion, which came out **tied**). Here we keep the **extra
concatenation heads** — the same 918-d concat (frozen square CNN 512 ⊕ averaged WS 406) fed to **logreg / RBF-SVM /
k-NN** — to show the result is *head-sensitive*.

**Be explicit:** feature/early fusion (concatenate, then one classifier) is **not** the same operation as true
late-probability fusion (average two branches' probabilities), and region-aware fusion belongs to **FINAL 2.4**.

In [12]:
Xcnn=frozen_feats("square",1.0); Xws=ws_avg("mass"); concat=np.concatenate([Xcnn,Xws],axis=1)
print("concat dim:", concat.shape[1], "(512 CNN + 406 WS)")
for hn,fac in [("logreg",logreg_pipe),("RBF-SVM",svm_pipe),("Subspace kNN",sknn_pipe)]:
    r=holdout5(concat,m_y,m_g,m_strat,fac); show("feature/concat -> %s"%hn, r)
    record("Fusion concat -> %s"%hn, "mass", r, PROV["RERUN"], "context",
           "head-sensitivity of early fusion (headline owned by FINAL 2.3)", "CNN-only/2.8")
# single-branch references under the same protocol
rc=holdout5(Xcnn,m_y,m_g,m_strat,logreg_pipe); show("(ref) CNN square -> logreg", rc)
rw=holdout5(Xws ,m_y,m_g,m_strat,sknn_pipe);   show("(ref) WS averaged -> kNN", rw)
print("\nRead: early-fusion AUROC depends strongly on the head; compare against the single branches above.")

concat dim: 918 (512 CNN + 406 WS)
  feature/concat -> logreg                         AUROC 0.915±0.028 | acc 0.849±0.035 | F1 0.887 


  feature/concat -> RBF-SVM                        AUROC 0.902±0.032 | acc 0.815±0.066 | F1 0.868 


  feature/concat -> Subspace kNN                   AUROC 0.882±0.043 | acc 0.790±0.058 | F1 0.847 
  (ref) CNN square -> logreg                       AUROC 0.857±0.059 | acc 0.808±0.054 | F1 0.856 


  (ref) WS averaged -> kNN                         AUROC 0.726±0.027 | acc 0.701±0.067 | F1 0.781 

Read: early-fusion AUROC depends strongly on the head; compare against the single branches above.


**What happened / interpretation.** Early fusion is **head-sensitive**: a **logistic-regression** head on the
918-d concat is the strong one (clearly above the WS branch and at/above the CNN branch), whereas the **Subspace
k-NN** head on the same concat sits ~at the CNN branch — which is exactly why FINAL 2.3's k-NN headline concat
looked like "no clear fusion gain". For completeness, the true **late-probability** fusion (averaging the two
branches' probabilities) is reported in **FINAL 2.3** (≈ 0.857 AUROC, tied with the k-NN concat); it is a
*different* operation and is not recomputed here. **Stable?** Mass bands are wide (n_test≈24); treat ranking
cautiously. **Why appendix:** this nuance (the fusion verdict depends on the classifier) is useful context, but the
*headline* fusion comparison is owned by FINAL 2.3 and the region-aware fusion by FINAL 2.4, so the extra heads
live here.

## 14. Superseded scalar controls

No additional single-seed control rows are registered here. The appendix summary below focuses on recomputed rows
and compact appendix references that support the final method choices.

In [ ]:
print("No extra single-seed control rows are registered in this appendix summary.")

**Protocol lessons (why the FINAL notebooks look the way they do).**
1. **Learned-feature CV is secondary.** For CNN-based branches, the report leads with repeated held-out AUROC and uses
   group-aware CV only as a paper-comparison or stability diagnostic.
2. **Split sensitivity matters.** The mass test set is small, so FINAL notebooks report **5-seed mean ± std** and lead
   with **AUROC** on the imbalanced mass task.
3. **Fold-local fitting.** Scalers, PCA, feature selection, and trainable feature extractors are fitted inside the
   relevant training split only.
4. **Determinism.** cuDNN/PyTorch nondeterminism is pinned (`set_seed`, deterministic flags) so reruns are as stable as
   the model class allows.

## 15. Final appendix summary table

Everything in this notebook in one place — recomputed reruns and appendix-reference rows together — so it is easy
to see what belongs in the report's screening table (the `rerun_final_protocol` rows of FINAL 2.3/2.4) and what
stays appendix-only.

In [ ]:
res = pd.DataFrame(ROWS)
res["AUROC"] = res.apply(lambda r: ("%.3f ± %.3f"%(r["auroc"],r["auroc_sd"])) if pd.notna(r.get("auroc_sd")) else
                         (("%.3f (ref)"%r["auroc"]) if pd.notna(r["auroc"]) else "—"), axis=1)
tab = res[["method","task","AUROC","provenance","status","reason","source"]]
res.to_csv(OUT/"appendix_methods_summary.csv", index=False)
import pandas as _pd; _pd.set_option("display.max_colwidth", 46); _pd.set_option("display.width", 240); _pd.set_option("display.max_rows", None)
display(tab)
print("(showing all %d appendix rows; the same table is also saved to the CSV below)" % len(tab))
# provenance breakdown
print("\nprovenance counts:", res["provenance"].value_counts().to_dict())
print("status counts    :", res["status"].value_counts().to_dict())
mres=res[(res["task"]=="mass") & (res["provenance"]=="rerun_final_protocol") & res["auroc"].notna()].sort_values("auroc")
fig,ax=plt.subplots(figsize=(9,max(3,0.42*len(mres))))
ax.barh(mres["method"].str.slice(0,46),mres["auroc"],xerr=mres["auroc_sd"],capsize=2,color="steelblue")
ax.axvline(0.5,ls=":",color="k"); ax.set_xlim(0.45,1.0); ax.set_xlabel("AUROC (mass, 5-seed mean±std)")
ax.set_title("Appendix recomputed methods — mass (AUROC-led)"); ax.grid(axis="x",alpha=.3)
plt.tight_layout(); plt.savefig(OUT/"appendix_mass_auroc.png",dpi=150,bbox_inches="tight"); plt.show()
print("saved:", OUT/"appendix_methods_summary.csv")

## 16. Closing note — how this appendix supports the report

This appendix exists so the main notebooks can stay focused:

- **It documents breadth.** Richer J6 summaries, full-flatten/PCA, the energy grid, the J-sweep, grid+Cohen's d
  hybrids, tensor-distance classifiers, extra CNN crops/heads, and extra fusion heads were all genuinely tried —
  recorded here as supporting context instead of crowding FINAL 2.3/2.4.
- **It keeps negative controls visible.** Tensor-distance and most J6 "unflattening" variants gave **no gain**; the
  energy grid **fails on mass**; normalisation is **seed-dependent**. These results explain why the report focuses on
  the stronger adopted methods.
- **It justifies the narrow main story.** The recurring signal — *preserve the spatial map and learn over it* —
  is what made **FINAL 2.3** (full-patch WS→CNN precursor) and **FINAL 2.4** (region-aware WS→CNN) the methods
  worth reporting. Everything that pointed there but didn't win lives here.

**Status:** appendix / supporting evidence. The adopted methods remain in FINAL 2.1–2.4; nothing here supersedes
them.